# 06 Geography: maps of IT growth by metro, and are the QCEW metro codes stable?

Stream B3. Needs `data/processed/metro_panel.csv` (`python build_metro_panel.py`), the Census Gazetteer centroids
(`python collect.py --sources gazetteer`) and, for section 3, `data/raw/bls_qcew_areas_it_industries.csv`.

**Limits to keep in view**
- IT = NAICS 5415 (IT consulting and computer systems design). Software publishers and big tech are *not* in it.
- IT employment is withheld (suppressed) for Chicago, Houston, Atlanta, Minneapolis and St. Louis (and others); on the employment map they appear as grey crosses, **not as zero growth**.
- Positions are internal points of the 2023 CBSA delineation (Gazetteer). The outline is a hand-simplified drawing aid, not an official boundary.
- Only the lower 48 states are drawn; Alaska, Hawaii and Puerto Rico metros are counted in the footnote.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "build_master.py").exists())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)      # the project's data paths are relative to the repo root
from collectors.common import RAW, PROCESSED
from analysis import geo

FIG = ROOT / "reports" / "figures"
if not (RAW / "census_gazetteer_cbsa.csv").exists():
    raise SystemExit("Run `python collect.py --sources gazetteer` first (Census Gazetteer centroids).")
G = geo.load_geo_panel()
cov = geo.coverage_report(G)
print(cov["metros"], "metros in the panel;", cov["drawn"], "drawn,", cov["outside_lower48"], "outside the lower 48,",
      cov["without_centroid"], "without a centroid", cov["without_centroid_names"][:10])

## 1. IT employment growth 2019 to 2024
Bubble area = IT jobs in 2019, colour = growth (blue up, red down, white 0). Grey crosses: IT employment unusable (withheld in 2019 and/or 2024).

In [ ]:
paths = geo.make_maps(G, FIG)
display(Image(filename=str(paths[0])))

## 2. IT establishment growth 2019 to 2024
Establishment counts are never withheld, so the five big missing metros are on this map. A hollow black outline means the 2019 job count is unknown, so the bubble has a fixed small size; where only the 2024 count is withheld (Chicago) the 2019 level still sizes the bubble.

In [ ]:
display(Image(filename=str(paths[1])))

In [ ]:
# The metros with no IT employment outcome: biggest first, and what we know about them anyway
miss = G[G["emp_basis"].eq("none")].nlargest(10, "emp_total_2019")
cols = [c for c in ["metro_name", "emp_total_2019", "emp_it_2019_reported", "estabs_it_2019", "estab_growth_pct"] if c in miss]
miss[cols].round(1)

## 3. Are the QCEW metro codes the same in 2019 and 2024?
QCEW codes metros `C####` (C + the first four digits of the CBSA code). The question is whether the set of codes changed with a new delineation
(new metros, merged or dropped ones). Two views: **any industry** (a code counts as present if it has any row that year) and **industry 10, all private employment**
(total employment is reported for every real area, so this is the cleaner test of the delineation itself).

In [ ]:
a = pd.read_csv(RAW / "bls_qcew_areas_it_industries.csv", dtype=str, usecols=["area_fips", "year", "industry_code"], low_memory=False)
a["year"] = a["year"].astype(int)
stab_any = geo.code_stability(a)
stab_tot = geo.code_stability(a, industry="10")
names = {**dict(zip(G["cbsa_code"], G["metro_name"])), **{}}
yrs = [c for c in stab_tot.columns if c.startswith("y") and c[1:].isdigit()]
print("metro codes per year (industry 10):", {c[1:]: int(stab_tot[c].sum()) for c in yrs})
print("in both 2019 and 2024:", int(((stab_tot["only_in"] == "") & stab_tot["y2019"] & stab_tot["y2024"]).sum()),
      "| only 2019:", int((stab_tot["only_in"] == "2019").sum()), "| only 2024:", int((stab_tot["only_in"] == "2024").sum()))
one = geo.only_one_year(stab_tot, names)
one.to_csv(PROCESSED / "qcew_metro_code_stability.csv", index=False)
one[["area_fips", "cbsa_code", "only_in", "name", *yrs]] if len(one) else print("No metro code is present in only one of the two years.")

In [ ]:
# Same check with any industry (should be a subset of the above plus codes that only have IT rows in one year)
only_any = geo.only_one_year(stab_any, names)
extra = sorted(set(only_any["area_fips"]) - set(one["area_fips"]))
print(len(only_any), "codes in only one year (any industry);", len(extra), "of them are not in the all-industries list:", extra[:15])

In [ ]:
# What a code check cannot see: a code that stays the same while its counties change. Heuristic: extreme total-employment change.
if "total_growth_pct" in G:
    flag = G.assign(abs_g=G["total_growth_pct"].abs()).nlargest(10, "abs_g")[["cbsa_code", "metro_name", "emp_total_2019", "total_growth_pct"]]
    display(flag.round(1))

In [ ]:
n19, n24 = int((one["only_in"] == "2019").sum()), int((one["only_in"] == "2024").sum())
print(f"- Answer (c): {'the metro code set is identical in 2019 and 2024' if n19 + n24 == 0 else f'{n19} metro codes appear only in 2019 and {n24} only in 2024 (list above and in data/processed/qcew_metro_code_stability.csv)'}.")
print("- Limit: this tests the codes, not the boundaries. The QCEW area file has no county list, so a metro that keeps its code but gains or loses counties is not detectable here;"
      " the extreme-total-growth table above is only a hint.")
print("- Consequence for the panel: metros present in one year only cannot have a 2019-to-2024 growth and are already excluded from the usable sets.")